# YATS for Quanto Users

You built Quanto; yats is its successor — same research philosophy, re-platformed. Every concept you know carries over; what changed is *where things live* and *how they run*. This notebook maps your Quanto muscle memory onto yats, with runnable cells.

**The one-paragraph version:** Parquet directories became QuestDB tables. The `quanto>` REPL and `scripts/*.py` became Dagster jobs (+ an MCP server so AI agents can drive the platform). Three overlapping risk layers became one 15-constraint engine wired into every path. Experiment specs, content-addressed IDs, qualification gates, shadow replay, and promotion tiers work exactly as you designed them — they just execute inside pipelines now. And options, which yats v1 deferred, are back via ThetaData with greeks + open interest that Quanto's iVolatility surface never matched.

## 1. Concept map — Quanto term → yats equivalent

| Quanto | yats | What changed |
|---|---|---|
| `quanto>` REPL (`cli/app.py`) | — removed | Dagster jobs + MCP tools + notebooks like this one |
| `scripts/ingest.py` + `configs/ingest/*.yml` | `ingest_alpaca`, `ingest_financialdatasets`, `ingest_thetadata` Dagster jobs | Vendors changed: Polygon → Alpaca; options → ThetaData |
| Raw Parquet `.quanto_data/raw/<vendor>/<domain>/` | QuestDB `raw_<vendor>_<domain>` tables (append-only, partitioned) | Same two-layer philosophy |
| `build-canonical` | `canonicalize` Dagster job → `canonical_*` tables | Adds streaming append (`stream_canonical`) + lineage columns |
| Feature sets (`core_v1`, registry) | Identical names, same registry pattern | Incremental recompute with `feature_watermarks` (no more full recompute every time) |
| `ExperimentSpec` YAML + sha256 `experiment_id` | `ExperimentSpec` dataclass, same content-addressed ID | Adds `extends`/`overrides` composition |
| `ExperimentRegistry` (directory walking) | Same `.yats_data/experiments/<id>/` tree **plus** `experiment_index` QuestDB table | `SELECT` instead of glob |
| `run-experiment` | `experiment_run` Dagster job | train → evaluate → index in one graph |
| `scripts/qualify_experiment.py` | `qualify` Dagster job | Same hard/soft gates incl. regime gates; auto shadow-replay for evidence |
| `scripts/promote_experiment.py` | `promote_job` | Tier ordering enforced research → candidate → production; production needs `managing_partner_ack` |
| `run-shadow` | `shadow_run` job / `ShadowEngine` | Writes unified `execution_log`/`execution_metrics` tables with a `mode` column (shadow/paper/live share schemas) |
| `research.risk.RiskConfig` + `ExecutionRiskEngine` + `execution/risk_checks.py` (3 layers) | **One** engine: `project_weights_full()` — 15 constraints, used by training, shadow, paper AND live | Plus risk *simulation mode*: spec-level overrides, shadow+sim only |
| `run_paper_scheduled` + `ops.yml` cron | `PaperTradingLoop` (independent process) + Dagster setup/teardown jobs + health sensor | Kill-switch state machine, heartbeat table, crash recovery |
| Streamlit dashboards | — removed | Dagster UI + QuestDB console (localhost:9000) + notebooks |
| Manifests (JSON per run) | `dagster_run_id` on every row + audit_trail table | Idempotent dedup keyed on run id |
| Options via iVolatility (surface) | ThetaData Terminal v3: chains, first-order greeks, IV, **open interest** | Deferred in v1, delivered 2026-07 |

## 2. Environment setup

One-time: `docker compose up -d` (QuestDB), `python scripts/bootstrap_db.py` (28 tables — **always after DDL changes, before any ingest**), and for options the ThetaData terminal:
```bash
# Java 21 required; bootstrap spawns a child JVM from PATH
export PATH="/opt/homebrew/opt/openjdk@21/bin:$PATH"
java -jar ~/thetadata/ThetaTerminalv3.jar --api-key $THETADATA_API_KEY   # serves http://127.0.0.1:25503/v3
```
Credentials live in a gitignored `.env` (see `.env.example`). The cell below loads it and wires paths.

In [ ]:
import os, sys, json, pathlib, urllib.parse, urllib.request
REPO = pathlib.Path.cwd()  # run this notebook from the repo root
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / 'pipelines'))
for line in (REPO / '.env').read_text().splitlines():
    if line.strip() and not line.startswith('#') and '=' in line:
        k, v = line.split('=', 1); os.environ.setdefault(k.strip(), v.split('#')[0].strip())

def qdb(sql: str):
    """Query QuestDB over HTTP — replaces Quanto's directory walking."""
    url = 'http://localhost:9000/exec?query=' + urllib.parse.quote(sql)
    with urllib.request.urlopen(url) as r:
        d = json.load(r)
    if 'error' in d: raise RuntimeError(d['error'])
    return d.get('dataset', []), [c['name'] for c in d.get('columns', [])]

rows, cols = qdb('SELECT count() FROM canonical_equity_ohlcv')
print('QuestDB up — canonical bars:', rows[0][0])

## 3. Data: ingest → canonical

In Quanto you ran `ingest --config ... --domain equity_ohlcv` then `build-canonical`. In yats each step is a Dagster job you can run from the UI (`uv run dagster dev`), the MCP `data.ingest` tool, or in-process like below. Note run-config lives in code/JSON, not per-run YAML files.

In [ ]:
from yats_pipelines.jobs.ingest_alpaca import ingest_alpaca
result = ingest_alpaca.execute_in_process(run_config={
    'ops': {'fetch_alpaca_bars': {'config': {
        'ticker_list': ['AAPL', 'MSFT'],
        'start_date': '2026-06-01', 'end_date': '2026-07-03',
    }}}
})
print('ingest success:', result.success)

In [ ]:
from yats_pipelines.jobs.canonicalize import canonicalize
result = canonicalize.execute_in_process(run_config={
    'ops': {'canonicalize_op': {'config': {'domains': ['equity_ohlcv']}}}
})
print('canonicalize success:', result.success)
rows, _ = qdb("SELECT symbol, count() FROM canonical_equity_ohlcv GROUP BY symbol LIMIT 5")
rows

**Muscle-memory alert:** there is no `--force`. Writes are idempotent — every row carries `dagster_run_id`, writers dedup before writing, and readers resolve duplicates with QuestDB's `LATEST ON`. Re-running a job is always safe.

## 4. Features: incremental, with watermarks

Quanto recomputed features from scratch each run. yats keeps `feature_watermarks` per symbol (plus `__universe__` and `__regime__` markers) and only computes new dates. Full recompute still exists as a separate job.

In [ ]:
from yats_pipelines.jobs.feature_pipeline_incremental import feature_pipeline_incremental
result = feature_pipeline_incremental.execute_in_process(run_config={
    'ops': {'feature_pipeline_incremental_op': {'config': {
        'universe': 'dev10',              # configs/universes/dev10.yml — same YAML shape as Quanto
        'regime_universe': ['SPY', 'QQQ'],
    }}}
})
print('incremental success:', result.success)
rows, _ = qdb('SELECT count() FROM feature_watermarks')
print('watermark rows:', rows[0][0])

## 5. Experiments — your spec, almost byte-for-byte

`ExperimentSpec` survived intact: content-addressed `experiment_id`, frozen dataclass, same policy names (`ppo`, `sac_*`, `sma`, `equal_weight`), same reward versioning. New: `extends`/`overrides` inheritance for sweep-style composition, and the registry ALSO writes an `experiment_index` row so you can SQL your history.

In [ ]:
from datetime import date
from pathlib import Path
from research.experiments.spec import ExperimentSpec, CostConfig
from research.experiments import registry

spec = ExperimentSpec(
    experiment_name='guide_demo_ppo',
    symbols=('AAPL', 'MSFT'),
    start_date=date(2025, 1, 2), end_date=date(2026, 6, 30),
    interval='daily',
    feature_set='core_v1',
    policy='ppo',
    policy_params={'total_timesteps': 2000, 'n_steps': 128, 'batch_size': 32, 'reward_version': 'v1'},
    cost_config=CostConfig(transaction_cost_bp=5.0),
    seed=7,
)
exp_id = registry.create(spec, data_root=Path('.yats_data'))
print('experiment_id:', exp_id[:16], '… (same sha256-of-canonical-JSON scheme as Quanto)')

In [ ]:
from yats_pipelines.jobs.experiment_run import experiment_run
cfg = {'experiment_id': exp_id, 'data_root': '.yats_data'}
ops = ['load_experiment_spec', 'fetch_features', 'train_policy', 'evaluate_experiment', 'update_experiment_index']
result = experiment_run.execute_in_process(run_config={'ops': {op: {'config': cfg} for op in ops}})
print('experiment success:', result.success)
m = json.loads((Path('.yats_data/experiments') / exp_id / 'evaluation/metrics.json').read_text())
{k: m['performance'][k] for k in ('sharpe', 'max_drawdown')}

In [ ]:
# Your directory-walking days are over:
rows, cols = qdb("SELECT experiment_id, sharpe, win_rate FROM experiment_index LATEST ON timestamp PARTITION BY experiment_id")
[dict(zip(cols, r)) for r in rows[:5]]

## 6. Qualify → promote

Same gates you designed (hard/soft/execution/regime; soft failures warn, don't block). Differences: it's a Dagster job, the replay for execution evidence runs under the **production** risk config automatically, and promotion enforces tier order — `research → candidate → production`, with `managing_partner_ack=True` required for production (the flag is non-bypassable in the MCP tool).

In [ ]:
from yats_pipelines.jobs.qualify import qualify
BASELINE = exp_id  # self-baseline for demo; normally latest:name-style resolution via experiment_index
cfg = {'candidate_id': exp_id, 'baseline_id': BASELINE, 'data_root': '.yats_data'}
ops = ['load_candidate_spec', 'load_baseline_and_metrics', 'evaluate_gates', 'update_experiment_index']
result = qualify.execute_in_process(run_config={'ops': {op: {'config': cfg} for op in ops}})
report = json.loads((Path('.yats_data/experiments') / exp_id / 'promotion/qualification_report.json').read_text())
print('passed:', report['passed'])

In [ ]:
from yats_pipelines.jobs.promote import promote_job
ops = ['verify_qualification', 'check_promotion_gates', 'write_promotion']
for tier in ('research', 'candidate'):
    cfg = {'experiment_id': exp_id, 'target_tier': tier,
           'promotion_reason': 'guide demo', 'promoted_by': 'notebook', 'data_root': '.yats_data'}
    r = promote_job.execute_in_process(run_config={'ops': {op: {'config': cfg} for op in ops}})
    print(tier, '->', r.success)
rows, _ = qdb("SELECT experiment_id, tier FROM promotions")
rows[-4:]

## 7. Shadow, paper, live — one table family

Quanto wrote `metrics_sim.json` per replay in its own format. yats writes shadow, paper, and live activity to the **same** `execution_log` / `execution_metrics` tables, distinguished by a `mode` column — so a query comparing shadow behaviour to paper behaviour is a `WHERE mode IN (...)` away. Filesystem artifacts (`steps.jsonl`, `summary.json`, `execution_metrics.json`) still exist per run.

In [ ]:
rows, cols = qdb("SELECT mode, count() FROM execution_metrics GROUP BY mode")
[dict(zip(cols, r)) for r in rows]

### Paper trading

`run_paper_scheduled` + cron became a long-running `PaperTradingLoop` process (Dagster only does setup/teardown + a health sensor). What you get that Quanto didn't have wired: the full 15-constraint risk engine pre-order **and** post-fill, a kill-switch state machine (`trading → halting → halted → resuming`) with `daily_loss`/`trailing_drawdown`/broker-failure triggers, a heartbeat table, and crash recovery that reconstructs positions from fills and reconciles pending orders against Alpaca by broker ID.

```python
from research.execution.paper_trading import PaperTradingLoop, PaperTradingConfig
loop = PaperTradingLoop(PaperTradingConfig(
    experiment_id=exp_id, symbols=['AAPL', 'MSFT'], initial_cash=100_000.0))
loop.start()   # blocking; signals arrive via loop.add_signals([...]) — Signal shape unchanged from Quanto
```
Risk decisions are logged to `risk_decisions` (e.g. a 150% target gets `size_reduce`d to the 20% symbol cap — try it).

## 8. Options — back, and better than iVolatility

Quanto ingested an options surface; yats v1 deferred options entirely; as of 2026-07 they're back via **ThetaData Terminal v3** (local process, port 25503). Chains with bid/ask, IV, first-order greeks and open interest land in `raw_thetadata_options_chain` → `canonical_options_chain` through the same two-layer model.

In [ ]:
from yats_pipelines.jobs.ingest_thetadata import ingest_thetadata   # requires the terminal running
result = ingest_thetadata.execute_in_process(run_config={
    'ops': {'fetch_thetadata_options': {'config': {'underlyings': ['AAPL'], 'expiry_window_days': 14}}}
})
print('options ingest:', result.success)
rows, cols = qdb("SELECT underlying, strike, right, delta, iv, open_interest FROM canonical_options_chain WHERE open_interest > 0 LIMIT 5")
[dict(zip(cols, r)) for r in rows]

## 9. The MCP layer (the thing Quanto never had)

`src/` hosts a TypeScript MCP server exposing 72 tools (`data.*`, `features.*`, `experiment.*`, `qualify.*`, `promote.*`, `risk.*`, `execution.*`, `monitor.*`, …) with role-based permissions, rate limits on destructive tools, and an append-only `audit_trail` of every invocation. It exists so AI agents (or you, via any MCP client) can drive everything this notebook does. You don't need it for hands-on work — everything here used the Python layer directly — but it's the intended front door for automation.

## 10. Gotchas for Quanto muscle memory

1. **Bootstrap before ingest.** QuestDB's ILP will happily auto-create a table with the wrong schema if you ingest before `scripts/bootstrap_db.py`. Always bootstrap after pulling DDL changes.
2. **No `--force`, no re-run anxiety** — idempotency via `dagster_run_id` dedup + `LATEST ON` reads.
3. **`interval='daily'` only**, same as Quanto v1.
4. **Timestamps are UTC everywhere**; when handling raw frames from psycopg2, normalize with `pd.to_datetime(..., utc=True)` before comparisons.
5. **Tests self-gate on the live stack**: `pytest tests` runs everything when QuestDB is up, skips the 13 live-DB tests when it's down. No `--ignore` flags.
6. **Your `.quanto_data` intuition maps directly**: `.yats_data/experiments/<id>/{spec,runs,evaluation,logs,promotion}` is the tree you already know; `promotions/<tier>/<id>.json` unchanged.
7. **Sweeps** exist (`experiment_sweep` job) and specs compose via `extends`/`overrides` — cleaner than Quanto's sweep_dimensions for one-off variants.